# 1. Kiểm tra dataset sau Cleaning

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit

# Xác định thư mục gốc IoT_final
current = Path.cwd().resolve()

ROOT = next(
    (
        folder
        for folder in [current, *current.parents]
        if (folder / "project").is_dir()
        and (folder / "result").is_dir()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục IoT_final")

# Đọc dữ liệu đã làm sạch trong C01
DATA_PATH = (
    ROOT / "result" / "preprocessing"
    / "C01_Data_Cleaning_ThanhDai"
    / "data" / "processed"
    / "RT_IOT2022_clean_ThanhDai.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy: {DATA_PATH}")

# Đường dẫn kết quả C02
RESULT_DIR = (
    ROOT / "result" / "preprocessing"
    / "C02_Data_Splitting_ThanhDai"
)

PROCESSED_DIR = RESULT_DIR / "data" / "processed"
TABLE_DIR = RESULT_DIR / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Đọc dataset
df = pd.read_csv(DATA_PATH)

target = "Attack_type"

print("=== DATASET SAU CLEANING ===")
print("Kích thước:", df.shape)
print("Số lớp:", df[target].nunique())
print("Missing:", df.isna().sum().sum())
print("Duplicate:", df.duplicated().sum())

display(df.head())

# Kiểm tra dữ liệu
assert df[target].nunique() == 12
assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0

### Nhận xét: Kiểm tra dữ liệu trước khi chia tập

**Kết quả dự kiến:**

- Dataset sau Cleaning gồm 117.909 bản ghi và 83 cột.
- Không còn bản ghi trùng hoàn toàn.
- Không phát hiện giá trị thiếu.
- Dữ liệu vẫn giữ đủ 12 lớp `Attack_type`.

**Kết luận:**

Dữ liệu đã hoàn thành giai đoạn làm sạch và
được kiểm tra trước khi chia tập.

# 2. Chia Train và Validation

In [ ]:
RANDOM_STATE = 42

splitter = StratifiedShuffleSplit(
    n_splits=1,
    train_size=0.70,
    test_size=0.15,
    random_state=RANDOM_STATE
)

# Lấy chỉ số của hai tập cần sử dụng
train_idx, val_idx = next(
    splitter.split(df, df[target])
)

# Tạo Train và Validation
train_df = df.iloc[train_idx].copy()
val_df = df.iloc[val_idx].copy()

# Đặt lại chỉ số dòng
train_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

print("=== KẾT QUẢ DATA SPLITTING ===")

print("Tổng dữ liệu gốc:", len(df))

print(
    f"Train: {len(train_df):,} mẫu "
    f"({len(train_df)/len(df)*100:.2f}%)"
)

print(
    f"Validation: {len(val_df):,} mẫu "
    f"({len(val_df)/len(df)*100:.2f}%)"
)

print(
    "Số mẫu chưa phân bổ:",
    len(df) - len(train_df) - len(val_df)
)

print("\nKích thước Train:", train_df.shape)
print("Kích thước Validation:", val_df.shape)

### Nhận xét: Chia dữ liệu

**Phương pháp:** StratifiedShuffleSplit

**Tham số:**
- Train: 70%.
- Validation: 15%.
- Phần chưa phân bổ: 15%.
- Random State: 42.

**Ý nghĩa:**

StratifiedShuffleSplit thực hiện lấy mẫu có phân tầng,
giúp tỷ lệ các lớp trong Train và Validation
gần với tỷ lệ trong dataset ban đầu.

Random State giúp tái lập kết quả khi chạy lại
trên cùng dữ liệu và cùng môi trường thư viện.

**Kết luận:**

Đã tạo hai tập dữ liệu Train và Validation
để chuẩn bị huấn luyện MLP.

Không sử dụng phần dữ liệu chưa phân bổ trong
quá trình thực nghiệm hiện tại.

# 3. Kiểm tra phân bố 12 nhãn sau khi chia

In [ ]:
original_counts = df[target].value_counts()
train_counts = train_df[target].value_counts()
val_counts = val_df[target].value_counts()

distribution = pd.DataFrame({
    "Original Count": original_counts,
    "Train Count": train_counts,
    "Validation Count": val_counts
}).fillna(0).astype(int)

distribution["Original (%)"] = (
    distribution["Original Count"] / len(df) * 100
)

distribution["Train (%)"] = (
    distribution["Train Count"] / len(train_df) * 100
)

distribution["Validation (%)"] = (
    distribution["Validation Count"] / len(val_df) * 100
)

distribution = distribution.sort_values(
    "Original Count",
    ascending=False
)

print("=== PHÂN BỐ NHÃN ===")

display(distribution.round(3))

# Kiểm tra độ lệch tỷ lệ
train_diff = (
    distribution["Train (%)"]
    - distribution["Original (%)"]
).abs()

val_diff = (
    distribution["Validation (%)"]
    - distribution["Original (%)"]
).abs()

print(
    "\nChênh lệch tỷ lệ Train lớn nhất:",
    f"{train_diff.max():.4f} điểm phần trăm"
)

print(
    "Chênh lệch tỷ lệ Validation lớn nhất:",
    f"{val_diff.max():.4f} điểm phần trăm"
)

print("\n=== KIỂM TRA LỚP HIẾM ===")

display(
    distribution.sort_values("Original Count").head(5)
)

# Lưu bảng
distribution.to_csv(
    TABLE_DIR / "C02_class_distribution.csv",
    encoding="utf-8-sig"
)

### Nhận xét: Phân bố nhãn sau Data Splitting

**Kết quả phân tích:**

- Train và Validation vẫn giữ đủ 12 lớp.
- Tỷ lệ các lớp gần với tỷ lệ trong dữ liệu ban đầu.
- Lớp DOS_SYN_Hping tiếp tục chiếm phần lớn dữ liệu.
- Các lớp NMAP_FIN_SCAN và Metasploit_Brute_Force_SSH
  vẫn có số lượng mẫu rất ít.

**Kết luận:**

Việc chia có phân tầng giúp duy trì tương đối
tỷ lệ nhãn nhưng không giải quyết mất cân bằng.

Tiếp tục sử dụng Macro-F1, Macro-Recall
và Classification Report khi đánh giá MLP.

# 4. Kiểm tra trùng lặp và Data Leakage giữa hai tập Train và Validation

In [ ]:
# Kiểm tra chỉ số gốc không bị chồng lấn
overlap_indices = np.intersect1d(train_idx, val_idx)

print("=== KIỂM TRA CHỒNG LẤN ===")

print(
    "Số chỉ số xuất hiện ở cả hai tập:",
    len(overlap_indices)
)

# Kiểm tra duplicate nội bộ
print(
    "Duplicate trong Train:",
    train_df.duplicated().sum()
)

print(
    "Duplicate trong Validation:",
    val_df.duplicated().sum()
)

# Kiểm tra mẫu trùng bằng mã băm toàn bộ cột
train_hash = pd.util.hash_pandas_object(
    train_df,
    index=False
)

val_hash = pd.util.hash_pandas_object(
    val_df,
    index=False
)

overlap_hashes = set(train_hash).intersection(
    set(val_hash)
)

print(
    "Số mã băm bản ghi trùng giữa hai tập:",
    len(overlap_hashes)
)

# Kiểm tra các lớp
print(
    "\nSố lớp Train:",
    train_df[target].nunique()
)

print(
    "Số lớp Validation:",
    val_df[target].nunique()
)

# Đảm bảo không có vấn đề cơ bản
assert len(overlap_indices) == 0
assert train_df.duplicated().sum() == 0
assert val_df.duplicated().sum() == 0
assert len(overlap_hashes) == 0
assert train_df[target].nunique() == 12
assert val_df[target].nunique() == 12

print("\nHoàn thành kiểm tra!")

### Nhận xét: Kiểm tra Data Leakage

**Kết quả kiểm tra:**

- Không có bản ghi nào thuộc đồng thời Train và Validation
  theo chỉ số dòng gốc.
- Không còn bản ghi trùng hoàn toàn trong mỗi tập.
- Không phát hiện mã băm bản ghi trùng giữa hai tập.
- Cả hai tập giữ đủ 12 lớp nhãn mục tiêu.

**Kết luận:**

Kết quả cho thấy không có hiện tượng chồng lấn
bản ghi giống hệt nhau giữa Train và Validation.

Tuy nhiên, kiểm tra này chưa loại trừ mọi
nguy cơ Data Leakage, chẳng hạn các phiên
kết nối liên quan xuất hiện ở hai tập.

Việc mã hóa, chuẩn hóa và lựa chọn đặc trưng
phải chỉ được fit trên Train trong giai đoạn
tiền xử lý tiếp theo.

# 5. Lưu Train và Validation

In [ ]:
train_path = PROCESSED_DIR / "train.csv"
val_path = PROCESSED_DIR / "validation.csv"

train_df.to_csv(
    train_path,
    index=False
)

val_df.to_csv(
    val_path,
    index=False
)

print("=== ĐÃ LƯU DATASET ===")

print("Train:", train_path)
print("Validation:", val_path)

print("\n=== KIỂM TRA FILE ===")

print("Train tồn tại:", train_path.exists())
print("Validation tồn tại:", val_path.exists())

# Lưu thông tin cấu hình chia dữ liệu
import json

split_info = {
    "source_file": DATA_PATH.name,
    "random_state": RANDOM_STATE,
    "method": "StratifiedShuffleSplit",
    "stratify_column": target,
    "train_ratio": 0.70,
    "validation_ratio": 0.15,
    "unallocated_ratio": 0.15,
    "original_rows": int(len(df)),
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "unallocated_rows": int(
        len(df) - len(train_df) - len(val_df)
    )
}

with open(
    RESULT_DIR / "split_info.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(split_info, f, indent=4, ensure_ascii=False)

print("\nĐã lưu cấu hình chia dữ liệu.")

### Nhận xét: Kết quả Data Splitting

**Kết quả thực hiện:**

- Đã xuất tập Train thành `train.csv`.
- Đã xuất tập Validation thành `validation.csv`.
- Đã lưu cấu hình chia tập trong `split_info.json`.
- Dữ liệu vẫn giữ nguyên kiểu giá trị ban đầu.

**Kết luận:**

Hai tập Train và Validation sẵn sàng
cho việc xây dựng pipeline tiền xử lý.

Chưa thực hiện One-Hot Encoding, Scaling,
Feature Selection hoặc Oversampling.